In [ ]:
import psycopg2
from psycopg2 import sql, extras
import logging
import csv
import os
import json


In [ ]:
# Set up basic logging to see success/failure messages
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

In [ ]:
DB_PARAMS = {
    'dbname': 'thecall',
    'user': 'postgres',        
    'password': 'password',    
    'host': 'localhost',           
    'port': '5432'                 
}

In [ ]:
def connect_to_db(params):
    """Establishes a connection to the PostgreSQL database."""
    conn = None
    try:
        logging.info("Attempting to connect to PostgreSQL database...")
        conn = psycopg2.connect(**params)
        # Set autocommit to False so we can control transactions (needed for UPDATE)
        conn.autocommit = False 
        logging.info("Connection successful.")
        return conn
    except psycopg2.Error as e:
        logging.error(f"Error connecting to the database: {e}")
        return None

In [ ]:
def collect_data_from_db(conn, table_schema, table_name):
    full_table_name = f"{table_schema}.{table_name}"
    try:
        with conn.cursor() as cur:
            # Fetch filename and folder to match with local files
            query = sql.SQL("SELECT id, json_raw FROM {}.{} WHERE json_raw is NULL").format(
                sql.Identifier(table_schema),
                sql.Identifier(table_name)
            )
            cur.execute(query)
            return cur.fetchall()
    except psycopg2.Error as e:
        logging.error(f"Error fetching data: {e}")
        return []

In [ ]:
def update_dbs(conn, table_schema, table_name, results):
    if not results:
        logging.info("No results to update.")
        return

    try:
        with conn.cursor() as cur:
            # Query to update json_unified
            update_query = """

            """
            
            extras.execute_values(cur, update_query, results)
            conn.commit()
            logging.info(f"Successfully updated {len(results)} rows in the database.")
            
    except psycopg2.Error as e:
        conn.rollback()
        logging.error(f"Database update failed: {e}")

In [ ]:
def main():
    conn = connect_to_db(DB_PARAMS)
    if conn:
        try:
            table_schema = 'articles'
            table_name = 'filelist'

            # 1. Fetch existing entries
            all_data = collect_data_from_db(conn, table_schema, table_name)
            
            if not all_data:
                logging.warning("No data retrieved from DB. Exiting.")
                return

            # 2. Batch update the DB
            update_dbs(conn, table_schema, table_name, results)
            
        finally:
            if conn:
                conn.close()
                logging.info("Database connection closed.")

In [ ]:
if __name__ == '__main__':
    main()